In [1]:
import os
import pandas as pd
from spectral.io import envi
from spectral.io.envi import read_envi_header
from glob import glob
import numpy as np

os.chdir('/store/carroll/sbgplants/')

In [2]:
# file paths
raw = 'data/raw'

fid_fol = '/store/carroll/col/data/2018/raw/L1'

out_folder = 'data/out_csv'

table = 'granule'

In [ ]:
# load relevant data
obs = glob(os.path.join(fid_fol, '*/*_rdn_obs_ort.hdr'))
fids = [x.split('/')[-1].removesuffix('_rdn_obs_ort.hdr') for x in obs]
dates = [x.split('_')[1] for x in fids]

In [ ]:
# get a list of times from obs data
times = []
for fp in obs:
    time = envi.open(fp).open_memmap()[...,-2].copy()
    time[time==-9999] = np.nan
    time = np.nanmean(time)
    times.append(float(time))

In [ ]:
times

In [ ]:
# prepare output table

out_table = pd.DataFrame(index=range(len(fids)))

out_table['granule_id'] = None
out_table['sensor_camp_id'] = 'NEON Imaging Spectrometer'
out_table['acquistion_start_time'] = times
out_table['flightline_id'] = fids
out_table['acquistion_date'] = dates
out_table['granule_rad_url'] = 'https://data.ess-dive.lbl.gov/view/doi%3A10.15485%2F1617204'
out_table['granule_refl_url'] = None

# to populate directly from raster asap
out_table['cloudy_conditions'] = None
out_table['gsd'] = None # ?

out_table

In [36]:
# prepare & populate out table
out_table = pd.DataFrame(columns=schema['column_name'].unique(), index=range(len(flightlines)))

out_table['sensor_camp_id'] = sensor_camp_id
out_table['acquisition_time'] = times # format? UTC
out_table['acquisition_date'] = dates # format?
out_table['doi_url'] = '10.15485/1617204'
out_table['cloudy_conditions'] = pd.NA # where to get the NEON red/yellow/green classification per flightline? And then how to translate that to boolean?
out_table['flightline_id'] = flightlines # to be replaced with uuid but useful for now
    # yellow = 10-50% cloud cover (cloudy_conditions==T)
    # green < 10 % cloud cover (cloudy_conditions==F)
    # from the post flight report I know that only 7 (or 8?)of all of the flights were yellow, rest were green.  And all yellow flights occured on 6/12 or 6/13. But unclear which flightlines were yellow
    # I can probably figure it out by looking at the radiance + figure 13 but it would be nicer if I could just find the metadata...
out_table

,sensor_camp_id,acquisition_time,acquisition_date,doi_url,cloudy_conditions,flightline_id
0,NEON Imaging Spectrometer,15.852367,20180612,10.15485/1617204,<NA>,NIS01_20180612_154959
1,NEON Imaging Spectrometer,15.932541,20180612,10.15485/1617204,<NA>,NIS01_20180612_155442
2,NEON Imaging Spectrometer,16.005730,20180612,10.15485/1617204,<NA>,NIS01_20180612_155857
3,NEON Imaging Spectrometer,16.086092,20180612,10.15485/1617204,<NA>,NIS01_20180612_160340
4,NEON Imaging Spectrometer,16.169357,20180612,10.15485/1617204,<NA>,NIS01_20180612_160819
5,NEON Imaging Spectrometer,16.258728,20180612,10.15485/1617204,<NA>,NIS01_20180612_161352
6,NEON Imaging Spectrometer,16.361725,20180612,10.15485/1617204,<NA>,NIS01_20180612_161935
7,NEON Imaging Spectrometer,16.498104,20180612,10.15485/1617204,<NA>,NIS01_20180612_162711
8,NEON Imaging Spectrometer,16.650967,20180612,10.15485/1617204,<NA>,NIS01_20180612_163536
9,NEON Imaging Spectrometer,16.773827,20180612,10.15485/1617204,<NA>,NIS01_20180612_164336


In [37]:
# check data types
out_table.dtypes

sensor_camp_id        object
acquisition_time     float64
acquisition_date      object
doi_url               object
cloudy_conditions     object
flightline_id         object
dtype: object

In [38]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)

fp_out

'/store/carroll/sbgplants/data/out_csv/flightline.csv'